In [ ]:
# =============================================================================
# 🧹 NETTOYAGE SYSTÈME (SSH) - EXÉCUTER EN PREMIER!
# =============================================================================

import os, gc, shutil, glob

def quick_cleanup():
    """Nettoyage rapide avant exécution."""
    print("🧹 NETTOYAGE...")
    gc.collect()
    
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            mem = torch.cuda.memory_allocated() / 1e9
            print(f"✅ CUDA: {mem:.2f} GB alloué")
    except: pass
    
    for pattern in ['**/__pycache__', '**/.ipynb_checkpoints']:
        for p in glob.glob(pattern, recursive=True):
            try: shutil.rmtree(p)
            except: pass
    
    total, used, free = shutil.disk_usage('/')
    print(f"💾 Espace: {free/1e9:.1f} GB libre")
    print("✅ Prêt!")

quick_cleanup()

In [ ]:
# =============================================================================
# 📦 IMPORTS & CONFIGURATION
# =============================================================================
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression

# Gradient Boosting
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️ Device: {device}')
print(f'🔢 PyTorch version: {torch.__version__}')

## 📥 1. Chargement des Données

In [ ]:
# =============================================================================
# 📂 PATHS CONFIGURATION - CORRIGÉS pour fichiers existants
# =============================================================================
DATA_DIR = './data'
EMB_DIR = os.path.join(DATA_DIR, 'embeddings')

# File paths - FICHIERS RÉELLEMENT EXISTANTS
paths = {
    # Embeddings CamemBERT (768d ou 2304d multi-layer)
    'train_embeddings': os.path.join(EMB_DIR, 'X_train_text_embeddings.npy'),
    'kaggle_embeddings': os.path.join(EMB_DIR, 'X_kaggle_text_embeddings.npy'),
    # Ou multi-layer (meilleur)
    'train_embeddings_ml': os.path.join(EMB_DIR, 'X_train_multilayer_embeddings.npy'),
    'kaggle_embeddings_ml': os.path.join(EMB_DIR, 'X_kaggle_multilayer_embeddings.npy'),
    # Features structurées (CSV de feature_engineering.ipynb)
    'train_features': os.path.join(DATA_DIR, 'train_features.csv'),
    'kaggle_features': os.path.join(DATA_DIR, 'test_features.csv'),
    # Labels
    'y_train': os.path.join(DATA_DIR, 'y_train.npy'),
}

# Check files exist
for name, path in paths.items():
    if os.path.exists(path):
        print(f"✅ {name}: {path}")
    else:
        print(f"❌ {name}: {path} - MANQUANT")

In [ ]:
# =============================================================================
# 📊 LOAD DATA - Utilise les fichiers réels
# =============================================================================
def clean_array(arr):
    """Clean NaN/inf values and convert to float32."""
    arr = np.asarray(arr)
    arr = np.nan_to_num(arr, nan=0.0, posinf=1e6, neginf=-1e6)
    return arr.astype(np.float32)

# Load embeddings (utiliser multi-layer si disponible, sinon standard)
USE_MULTILAYER = os.path.exists(paths['train_embeddings_ml'])
if USE_MULTILAYER:
    X_train_emb = clean_array(np.load(paths['train_embeddings_ml']))
    X_kaggle_emb = clean_array(np.load(paths['kaggle_embeddings_ml']))
    print(f"📊 Using MULTILAYER embeddings (2304d)")
else:
    X_train_emb = clean_array(np.load(paths['train_embeddings']))
    X_kaggle_emb = clean_array(np.load(paths['kaggle_embeddings']))
    print(f"📊 Using standard embeddings (768d)")

# Load structured features from CSV (feature_engineering.ipynb output)
train_df = pd.read_csv(paths['train_features'])
kaggle_df = pd.read_csv(paths['kaggle_features'])

# ⭐ Features numériques les plus importantes (basées sur LightGBM feature importance)
# Ordonnées par importance décroissante selon feature_engineering.ipynb
TOP_FEATURES = [
    # TOP 6 features (importance > 500)
    'user_description_length',   # importance=736 ⭐
    'tweets_per_favourites',     # importance=699 ⭐
    'user_favourites_count',     # importance=646 ⭐
    'user_statuses_count',       # importance=639 ⭐
    'user_listed_count',         # importance=607 ⭐
    'listed_per_status',         # importance=553 ⭐
    # Log transforms (haute corrélation)
    'log_user_listed',           # corr=0.606 ⭐
    'log_user_statuses',         # corr=0.439
    'log_user_favourites',
    # Engagement features
    'total_engagement', 'log_total_engagement',
    'retweet_count', 'favorite_count', 'reply_count', 'quote_count',
    'log_retweet_count', 'log_favorite_count',
    # Binary user features (très discriminantes)
    'user_has_url',              # Observer=16%, Influencer=56%
    'user_has_banner',           # Observer=73%, Influencer=92%
    'user_has_location',         # Observer=58%, Influencer=75%
    'user_has_long_desc',
    'user_default_profile', 'user_default_profile_image',
    # Source device (très discriminant)
    'is_iphone', 'is_android', 'is_web', 'is_tweetdeck', 'is_bot_source',
    # Text features
    'tweet_length', 'word_count', 'uppercase_ratio',
    'hashtag_count', 'is_hashtag_heavy',
    'mention_count', 'is_mention_heavy',
    'emoji_count', 'is_emoji_heavy',
    'exclamation_count', 'question_count', 'has_multiple_exclamations',
    'url_count', 'has_url',
    # Content detection
    'is_reply', 'is_in_reply', 'is_reply_to_someone',
    'has_rt_qt', 'is_quote_status', 'has_quoted_status',
    'has_call_to_action', 'has_self_promotion', 'has_media_reference',
    # Entities
    'entities_hashtags', 'entities_urls', 'entities_mentions', 'entities_symbols',
    # Other
    'first_person_count', 'is_long_tweet', 'is_short_tweet'
]

# Filter to features that exist
available_features = [f for f in TOP_FEATURES if f in train_df.columns]
print(f"📊 Using {len(available_features)} structured features")

# Extract and clean features
X_train_feat = clean_array(train_df[available_features].fillna(0).values)
X_kaggle_feat = clean_array(kaggle_df[available_features].fillna(0).values)

# Normalize features
from sklearn.preprocessing import StandardScaler
feat_scaler = StandardScaler()
X_train_feat = feat_scaler.fit_transform(X_train_feat)
X_kaggle_feat = feat_scaler.transform(X_kaggle_feat)

# Load labels
y_full = np.load(paths['y_train'])
le = LabelEncoder()
y_full = le.fit_transform(y_full)

print(f"\n📊 Data Shapes:")
print(f"   Embeddings (train): {X_train_emb.shape}")
print(f"   Structured features (train): {X_train_feat.shape}")
print(f"   Labels: {y_full.shape}, Classes: {np.unique(y_full)}")
print(f"   Class distribution: {np.bincount(y_full)}")

## 🔧 2. Feature Engineering Avancé

Inspiré de Lab6 - Création de features combinées et interactions.

In [ ]:
# =============================================================================
# 🔧 COMBINE EMBEDDINGS + FEATURES
# =============================================================================

# Combine: embeddings + structured features
X_train_combined = np.hstack([X_train_emb, X_train_feat])
X_kaggle_combined = np.hstack([X_kaggle_emb, X_kaggle_feat])

print(f"📊 Combined features:")
print(f"   Train: {X_train_combined.shape}")
print(f"   Kaggle: {X_kaggle_combined.shape}")

# Calculate dimensions for the neural network
EMB_DIM = X_train_emb.shape[1]
FEAT_DIM = X_train_feat.shape[1]
TOTAL_DIM = X_train_combined.shape[1]

print(f"\n🔢 Dimensions:")
print(f"   Embedding dim: {EMB_DIM}")
print(f"   Feature dim: {FEAT_DIM}")
print(f"   Total dim: {TOTAL_DIM}")

In [ ]:
# =============================================================================
# 📊 TRAIN/VAL SPLIT
# =============================================================================
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_train_combined, y_full, 
    test_size=0.15, 
    random_state=SEED, 
    stratify=y_full
)

# Also split embeddings and features separately for multi-branch model
X_train_emb_split, X_val_emb_split = train_test_split(
    X_train_emb, test_size=0.15, random_state=SEED, stratify=y_full
)
X_train_feat_split, X_val_feat_split = train_test_split(
    X_train_feat, test_size=0.15, random_state=SEED, stratify=y_full
)

print(f"📊 Train/Val split:")
print(f"   Train: {X_train.shape}, Labels: {np.bincount(y_train)}")
print(f"   Val: {X_val.shape}, Labels: {np.bincount(y_val)}")

In [ ]:
# =============================================================================
# 📊 TRAIN/VAL SPLIT
# =============================================================================

# Split for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_all, y_full,
    test_size=0.15,
    random_state=SEED,
    stratify=y_full
)

# Also split the individual components for the neural network
(
    X_train_t, X_val_t,
    X_train_d, X_val_d,
    X_train_f, X_val_f,
    y_train_nn, y_val_nn
) = train_test_split(
    X_train_text, X_train_desc, X_train_feat, y_full,
    test_size=0.15,
    random_state=SEED,
    stratify=y_full
)

print(f"📊 Split sizes:")
print(f"   Train: {X_train.shape[0]}")
print(f"   Val: {X_val.shape[0]}")

## 🧠 3. Neural Network Architecture (Lab4 + Lab5 inspired)

Multi-branch architecture avec:
- Dropout régularisation (Lab4)
- Batch Normalization
- Residual connections (Lab5 Transformer concept)
- Multi-Sample Dropout (recommandé dans TODO.md)

In [ ]:
# =============================================================================
# 🧠 MULTI-BRANCH NEURAL NETWORK (Lab4 + Lab5 inspired)
# =============================================================================

class MultiSampleDropout(nn.Module):
    """Multi-Sample Dropout from Lab4 concepts - averages multiple dropout masks."""
    def __init__(self, dropout_rates=[0.1, 0.2, 0.3, 0.4, 0.5]):
        super().__init__()
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in dropout_rates])
    
    def forward(self, x):
        if self.training:
            return torch.mean(torch.stack([d(x) for d in self.dropouts]), dim=0)
        return x


class AttentionPooling(nn.Module):
    """Simple attention pooling for combining features (Lab5 inspired)."""
    def __init__(self, input_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(input_dim, input_dim // 4),
            nn.Tanh(),
            nn.Linear(input_dim // 4, 1),
            nn.Softmax(dim=1)
        )
    
    def forward(self, x):
        # x: [batch, num_features, feature_dim]
        weights = self.attention(x)  # [batch, num_features, 1]
        return (x * weights).sum(dim=1)  # [batch, feature_dim]


class InfluencerClassifierAdvanced(nn.Module):
    """
    Advanced Multi-Branch Classifier.
    
    Architecture:
    - Tweet branch: processes tweet embeddings
    - User branch: processes user description embeddings  
    - Feature branch: processes structured features
    - Fusion: concatenate + attention + classification head
    
    Regularization (Lab4):
    - Dropout between layers
    - Batch Normalization
    - Weight decay (in optimizer)
    """
    
    def __init__(self, tweet_dim=768, user_dim=768, feat_dim=100, 
                 hidden_dim=256, n_classes=2, dropout=0.3):
        super().__init__()
        
        # Tweet embedding branch
        self.tweet_branch = nn.Sequential(
            nn.Linear(tweet_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(inplace=True),
        )
        
        # User description branch
        self.user_branch = nn.Sequential(
            nn.Linear(user_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(inplace=True),
        )
        
        # Structured features branch
        self.feat_branch = nn.Sequential(
            nn.Linear(feat_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
        )
        
        # Fusion dimension
        fusion_dim = hidden_dim // 2 + hidden_dim // 2 + 32
        
        # Multi-Sample Dropout (Lab4 regularization)
        self.ms_dropout = MultiSampleDropout([0.1, 0.2, 0.3])
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, n_classes)
        )
        
        # Initialize weights (Lab4 best practice)
        self._init_weights()
    
    def _init_weights(self):
        """He initialization for ReLU networks."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, tweet, user, feat):
        # Process each branch
        t = self.tweet_branch(tweet)
        u = self.user_branch(user)
        f = self.feat_branch(feat)
        
        # Concatenate
        x = torch.cat([t, u, f], dim=1)
        
        # Apply multi-sample dropout
        x = self.ms_dropout(x)
        
        # Classification
        return self.classifier(x)


# Test the model
feat_dim = X_train_f.shape[1]
model = InfluencerClassifierAdvanced(
    tweet_dim=768, user_dim=768, feat_dim=feat_dim,
    hidden_dim=256, n_classes=2, dropout=0.3
)
print(f"📐 Model architecture:")
print(model)
print(f"\n📊 Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 🏋️ 4. Training Loop (Lab4 + Lab8 inspired)

Avec:
- AdamW optimizer avec weight decay (Lab8)
- Learning rate scheduling (OneCycleLR)
- Early stopping
- Gradient clipping

In [ ]:
# =============================================================================
# 📦 DATASET & DATALOADER
# =============================================================================

class InfluencerDataset(Dataset):
    """Custom dataset for multi-input model."""
    def __init__(self, tweet_emb, user_emb, features, labels=None):
        self.tweet = torch.from_numpy(tweet_emb).float()
        self.user = torch.from_numpy(user_emb).float()
        self.feat = torch.from_numpy(features).float()
        self.labels = None if labels is None else torch.from_numpy(labels).long()
    
    def __len__(self):
        return self.tweet.shape[0]
    
    def __getitem__(self, idx):
        if self.labels is None:
            return self.tweet[idx], self.user[idx], self.feat[idx]
        return self.tweet[idx], self.user[idx], self.feat[idx], self.labels[idx]


# Create datasets
batch_size = 64

train_dataset = InfluencerDataset(X_train_t, X_train_d, X_train_f, y_train_nn)
val_dataset = InfluencerDataset(X_val_t, X_val_d, X_val_f, y_val_nn)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        num_workers=2, pin_memory=True)

print(f"📊 DataLoaders created:")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")

In [ ]:
# =============================================================================
# 🏋️ TRAINING UTILITIES (Lab4 + Lab8 inspired)
# =============================================================================

def train_epoch(model, loader, optimizer, criterion, scheduler, device, max_grad_norm=1.0):
    """Train for one epoch with gradient clipping (Lab4)."""
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    
    for batch in loader:
        tweet, user, feat, labels = [b.to(device) for b in batch]
        
        optimizer.zero_grad()
        logits = model(tweet, user, feat)
        loss = criterion(logits, labels)
        
        loss.backward()
        
        # Gradient clipping (prevents exploding gradients - Lab4)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(loader), acc, f1


def evaluate(model, loader, criterion, device):
    """Evaluate model on validation set."""
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []
    
    with torch.no_grad():
        for batch in loader:
            tweet, user, feat, labels = [b.to(device) for b in batch]
            logits = model(tweet, user, feat)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            probs = F.softmax(logits, dim=1)
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(loader), acc, f1, np.array(all_probs)


def plot_training_history(history):
    """Plot training curves (Lab4 style)."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Loss
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss Curve')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(history['train_acc'], label='Train')
    axes[1].plot(history['val_acc'], label='Val')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy Curve')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # F1 Score
    axes[2].plot(history['train_f1'], label='Train')
    axes[2].plot(history['val_f1'], label='Val')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('F1 Score')
    axes[2].set_title('F1 Score Curve')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# 🚀 TRAINING LOOP
# =============================================================================

# Hyperparameters (Lab8 - optimized values)
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 0.01  # L2 regularization (Lab4)
PATIENCE = 5  # Early stopping patience

# Initialize model
model = InfluencerClassifierAdvanced(
    tweet_dim=768, user_dim=768, feat_dim=feat_dim,
    hidden_dim=256, n_classes=2, dropout=0.3
).to(device)

# Loss function with class weights (handle imbalance)
class_counts = np.bincount(y_train_nn)
class_weights = torch.tensor([1.0 / c for c in class_counts], dtype=torch.float32).to(device)
class_weights = class_weights / class_weights.sum() * 2  # Normalize
criterion = nn.CrossEntropyLoss(weight=class_weights)

# AdamW optimizer with weight decay (Lab8 - better than Adam for regularization)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Learning rate scheduler (OneCycleLR - state of the art)
total_steps = len(train_loader) * EPOCHS
scheduler = OneCycleLR(
    optimizer, 
    max_lr=LR,
    total_steps=total_steps,
    pct_start=0.1,  # Warmup
    anneal_strategy='cos'
)

print(f"🚀 Training configuration:")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning rate: {LR}")
print(f"   Weight decay: {WEIGHT_DECAY}")
print(f"   Class weights: {class_weights.cpu().numpy()}")

In [ ]:
# =============================================================================
# 🏃 RUN TRAINING
# =============================================================================

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [], 'val_acc': [],
    'train_f1': [], 'val_f1': []
}

best_val_acc = 0
best_val_f1 = 0
patience_counter = 0
best_model_state = None

print("\n" + "="*70)
print("🏋️ TRAINING STARTED")
print("="*70)

for epoch in range(1, EPOCHS + 1):
    # Train
    train_loss, train_acc, train_f1 = train_epoch(
        model, train_loader, optimizer, criterion, scheduler, device
    )
    
    # Evaluate
    val_loss, val_acc, val_f1, val_probs = evaluate(
        model, val_loader, criterion, device
    )
    
    # Store history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)
    
    # Print progress
    print(f"Epoch {epoch:2d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} F1: {train_f1:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}")
    
    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_val_acc = val_acc
        best_model_state = model.state_dict().copy()
        patience_counter = 0
        print(f"   ✅ New best model! Val F1: {val_f1:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n⏹️ Early stopping at epoch {epoch}")
            break

print("\n" + "="*70)
print(f"🏆 Best model - Val Accuracy: {best_val_acc:.4f}, Val F1: {best_val_f1:.4f}")
print("="*70)

# Restore best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)

In [ ]:
# Plot training history
plot_training_history(history)

## 🌳 5. Gradient Boosting Models (Ensemble Diversity)

In [ ]:
# =============================================================================
# 🌳 XGBOOST & LIGHTGBM (for ensemble)
# =============================================================================

# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# XGBoost
print("\n🌳 Training XGBoost...")
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    eval_metric='logloss',
    early_stopping_rounds=50,
    n_jobs=-1
)

xgb_model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_val_scaled, y_val)],
    verbose=False
)

xgb_preds = xgb_model.predict(X_val_scaled)
xgb_probs = xgb_model.predict_proba(X_val_scaled)
print(f"   XGBoost Val Accuracy: {accuracy_score(y_val, xgb_preds):.4f}")
print(f"   XGBoost Val F1: {f1_score(y_val, xgb_preds, average='macro'):.4f}")

# LightGBM
print("\n🌳 Training LightGBM...")
lgbm_model = LGBMClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)

lgbm_model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_val_scaled, y_val)],
    callbacks=[]
)

lgbm_preds = lgbm_model.predict(X_val_scaled)
lgbm_probs = lgbm_model.predict_proba(X_val_scaled)
print(f"   LightGBM Val Accuracy: {accuracy_score(y_val, lgbm_preds):.4f}")
print(f"   LightGBM Val F1: {f1_score(y_val, lgbm_preds, average='macro'):.4f}")

## 🎲 6. Ensemble (Weighted Voting)

In [ ]:
# =============================================================================
# 🎲 ENSEMBLE - WEIGHTED VOTING
# =============================================================================

# Get Neural Network predictions
_, nn_val_acc, nn_val_f1, nn_val_probs = evaluate(model, val_loader, criterion, device)

# Ensemble weights (based on validation F1)
weights = {
    'nn': nn_val_f1,
    'xgb': f1_score(y_val, xgb_preds, average='macro'),
    'lgbm': f1_score(y_val, lgbm_preds, average='macro')
}

# Normalize weights
total_weight = sum(weights.values())
weights = {k: v / total_weight for k, v in weights.items()}

print("\n🎲 Ensemble Weights:")
for name, w in weights.items():
    print(f"   {name}: {w:.4f}")

# Weighted average of probabilities
ensemble_probs = (
    weights['nn'] * nn_val_probs +
    weights['xgb'] * xgb_probs +
    weights['lgbm'] * lgbm_probs
)

ensemble_preds = np.argmax(ensemble_probs, axis=1)
ensemble_acc = accuracy_score(y_val, ensemble_preds)
ensemble_f1 = f1_score(y_val, ensemble_preds, average='macro')

print(f"\n🏆 Ensemble Results:")
print(f"   Accuracy: {ensemble_acc:.4f}")
print(f"   F1 Score: {ensemble_f1:.4f}")

# Compare all models
print("\n" + "="*50)
print("📊 MODEL COMPARISON")
print("="*50)
print(f"{'Model':<15} {'Accuracy':>10} {'F1 Score':>10}")
print("-"*35)
print(f"{'Neural Net':<15} {nn_val_acc:>10.4f} {nn_val_f1:>10.4f}")
print(f"{'XGBoost':<15} {accuracy_score(y_val, xgb_preds):>10.4f} {f1_score(y_val, xgb_preds, average='macro'):>10.4f}")
print(f"{'LightGBM':<15} {accuracy_score(y_val, lgbm_preds):>10.4f} {f1_score(y_val, lgbm_preds, average='macro'):>10.4f}")
print(f"{'Ensemble':<15} {ensemble_acc:>10.4f} {ensemble_f1:>10.4f}")
print("="*50)

## 📝 7. Generate Kaggle Submission

In [ ]:
# =============================================================================
# 📝 GENERATE KAGGLE SUBMISSION
# =============================================================================

# Prepare Kaggle data
X_kaggle_all_scaled = scaler.transform(X_kaggle_all)

# Create Kaggle dataset for NN
kaggle_dataset = InfluencerDataset(X_kaggle_text, X_kaggle_desc, X_kaggle_feat, labels=None)
kaggle_loader = DataLoader(kaggle_dataset, batch_size=batch_size, shuffle=False,
                           num_workers=2, pin_memory=True)

# Get predictions from all models
print("\n📝 Generating predictions...")

# Neural Network
model.eval()
nn_kaggle_probs = []
with torch.no_grad():
    for batch in kaggle_loader:
        tweet, user, feat = [b.to(device) for b in batch]
        logits = model(tweet, user, feat)
        probs = F.softmax(logits, dim=1)
        nn_kaggle_probs.extend(probs.cpu().numpy())
nn_kaggle_probs = np.array(nn_kaggle_probs)

# XGBoost & LightGBM
xgb_kaggle_probs = xgb_model.predict_proba(X_kaggle_all_scaled)
lgbm_kaggle_probs = lgbm_model.predict_proba(X_kaggle_all_scaled)

# Ensemble
ensemble_kaggle_probs = (
    weights['nn'] * nn_kaggle_probs +
    weights['xgb'] * xgb_kaggle_probs +
    weights['lgbm'] * lgbm_kaggle_probs
)
ensemble_kaggle_preds = np.argmax(ensemble_kaggle_probs, axis=1)

print(f"   Total predictions: {len(ensemble_kaggle_preds)}")
print(f"   Class distribution: {np.bincount(ensemble_kaggle_preds)}")

In [ ]:
# =============================================================================
# 💾 SAVE SUBMISSION
# =============================================================================

# Load test data for IDs
kaggle_df = pd.read_json('data/kaggle_test.jsonl', lines=True)
kaggle_df = pd.json_normalize(kaggle_df.to_dict(orient='records'))

# Get challenge IDs
ids = kaggle_df['challenge_id'].astype(int).values

# Create submission DataFrame
submission = pd.DataFrame({
    'ID': ids,
    'Prediction': ensemble_kaggle_preds
})

# Save
os.makedirs('submission', exist_ok=True)
submission_path = 'submission/submission_model_with_features.csv'
submission.to_csv(submission_path, index=False)

print(f"\n✅ Submission saved to: {submission_path}")
print(f"   Shape: {submission.shape}")
print(f"   Class distribution: {submission['Prediction'].value_counts().to_dict()}")
print("\n📊 Sample predictions:")
print(submission.head(10))

## 📈 8. Résumé et Prochaines Étapes

### Ce notebook combine:
1. **Lab4 Techniques**: Dropout, Weight Decay, Gradient Clipping
2. **Lab5 Techniques**: Transfer Learning via CamemBERT embeddings
3. **Lab6 Techniques**: Feature Engineering, interaction features
4. **Lab8 Techniques**: AdamW optimizer, Learning Rate Scheduling

### Améliorations possibles:
- Fine-tuning CamemBERT end-to-end (si GPU disponible)
- Pseudo-labeling sur les prédictions confiantes
- Cross-validation pour des poids d'ensemble plus robustes
- Test-Time Augmentation (TTA)

In [ ]:
# =============================================================================
# 💾 SAVE MODEL CHECKPOINT
# =============================================================================

os.makedirs('models', exist_ok=True)

# Save Neural Network
checkpoint = {
    'model_state_dict': model.state_dict(),
    'feat_dim': feat_dim,
    'best_val_acc': best_val_acc,
    'best_val_f1': best_val_f1,
    'weights': weights
}
torch.save(checkpoint, 'models/influencer_model_advanced.pt')

# Save XGBoost & LightGBM
import joblib
joblib.dump(xgb_model, 'models/xgb_model.joblib')
joblib.dump(lgbm_model, 'models/lgbm_model.joblib')
joblib.dump(scaler, 'models/scaler.joblib')

print("✅ All models saved!")